# Fig. 2 dynamics with selectable disk structure

Choose the paper disk, an equivalent `gas_migration.py` disk, or the `gas_profile.ipynb` disk in the configuration cell. All experiments retain 10 km paper drag, Jupiter, Saturn and a gapped softened-ring gravity model. Candidate secular-resonance curves are diagnostics. The saved 10 Myr reference is only displayed when its metadata matches the current configuration.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "src" / "accrediff").exists()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("AccreDiff repository root not found")
HERE = PROJECT_ROOT / "examples" / "Expansion_module" / "GasDrag_Type1"
sys.path.insert(0, str(HERE))
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from paper_fig2_experiment import make_disk, output_path, matches_configuration
from paper_fig2_disk import PaperFig2Disk
from paper_fig2_drag import Planetesimal, GasFlowPolicy, AerodynamicDrag
from paper_fig2_gravity import GapGravityConfig, PaperGapDiskGravity
from paper_fig2_events import ParticleBoundaryConfig
from paper_fig2_system import Fig2SystemConfig, IntegrationConfig
from paper_fig2_integrator import integrate_fig2
from paper_fig2_resonances import locate_secular_resonances
from paper_fig2_diagnostics import plot_fig2_style, default_mmr_locations

## Explicit planetesimal initial conditions and force configuration

Edit `initial_a_au` to change the number and initial semimajor axes of the planetesimals. `initial_e` may be one scalar shared by every planetesimal, or a tuple with one value per semimajor axis. Angles default to zero below and are generated with the correct length. The planetesimals remain massless test particles dynamically; `body` controls the physical radius and density used by gas drag.

In [ ]:
from accrediff.gas_migration import Gas_ModelConfig, Gas_DiskModel, Gas_UnitSystem
from paper_fig2_disk import GasMigrationDiskAdapter

gas_cfg = Gas_ModelConfig(
    Sigma0_cgs=2000.0,       # 1 AU 表面密度，g/cm²
    a_alpha1=1.0,            # 表面密度剖面的径向弯曲
    a_alpha2=0.5,            # 1 AU 处的对数斜率
    def_h_1=0.0335743,       # 1 AU 处 H/r
    beta=0.25,              # H/r 随半径变化的指数
    tau_decay_myr=1.0,      # 气盘消散时间，Myr
)
units = Gas_UnitSystem()
disk = GasMigrationDiskAdapter(Gas_DiskModel(gas_cfg, units), units)

experiment_label = "my_custom_disk"


In [ ]:
# --- Explicit planetesimal initial conditions ---
initial_a_au = (3.5, 3.2, 2.7, 2.3)       # one entry per planetesimal, AU
initial_e = 0.01                            # one shared eccentricity
# For individual values use, e.g.: initial_e = (0.01, 0.02, 0.05, 0.10)

n_planetesimals = len(initial_a_au)
system = Fig2SystemConfig(
    particle_a_au=tuple(initial_a_au),
    particle_e=initial_e,
    particle_omega_rad=(0.0,) * n_planetesimals,
    particle_mean_anomaly_rad=(0.0,) * n_planetesimals,
)

# --- Planetesimal physical properties used by aerodynamic drag ---
body = Planetesimal(radius_km=100, solid_density_cgs=3)
drag = AerodynamicDrag(body, GasFlowPolicy("paper_2016"), "paper_2016", 0.165)

# --- Disk gravity table and independent particle termination boundaries ---
gap_cfg = GapGravityConfig(
    gap_inner_au=4.5, gap_outer_au=11.0,
    outer_edge_au=50.0, softening_h=0.6,
    table_min_au=0.1, table_max_au=50.0, n_table=3072,
)
gravity = PaperGapDiskGravity(disk, gap_cfg)
boundaries = ParticleBoundaryConfig(
    inner_termination_au=0.3,
    outer_termination_au=13.0,
    event_check_interval_years=100.0,
)

print(f"Body mass: {body.mass_mearth:.3e} Earth masses")
print(f"Initial a: {system.particle_a_au} AU")
print(f"Initial e: {system.particle_e_values}")
print(f"Gap: {gap_cfg.gap_inner_au:g}–{gap_cfg.gap_outer_au:g} AU")
print("Jupiter MMRs:", default_mmr_locations(system.jupiter_a_au))

## Disk-induced candidate secular resonance locations

The matrix uses linear Laplace-Lagrange secular terms plus the ring-gravity circular-orbit precession rate. Roots near strong MMRs are diagnostic only because linear secular theory is invalid there. More than one root can exist at a time, and zero roots are allowed.

In [ ]:
resonance_times = np.linspace(0.0, 10.0, 121)
resonance_radii = np.linspace(1.5, 4.3, 360)
resonances = locate_secular_resonances(system, gravity, resonance_times,
                                       resonance_radii, n_nodes=128)
fig, ax = plt.subplots(figsize=(8, 4.7))
for roots_by_time, color, label in ((resonances.nu5_roots_au, "black", r"$\nu_5$ candidate"),
                                    (resonances.nu6_roots_au, "gray", r"$\nu_6$ candidate")):
    x = [t for t, roots in zip(resonance_times, roots_by_time) for _ in roots if t > 0]
    y = [root for t, roots in zip(resonance_times, roots_by_time) for root in roots if t > 0]
    ax.scatter(x, y, s=5, color=color, label=label)
for label, radius in default_mmr_locations(system.jupiter_a_au).items():
    ax.axhline(radius, color="gold", lw=0.8)
    ax.text(10, radius, label, ha="right", va="bottom", fontsize=7)
ax.set(xscale="log", xlim=(0.1, 10), ylim=(1.5, 4.3),
       xlabel=r"$t/T_{dep}$", ylabel="Candidate resonance radius (AU)",
       title="Secular roots from softened ring gravity: diagnostic only")
ax.grid(alpha=0.2)
ax.legend()
fig.tight_layout()
plt.show()
for t in (1, 2, 5):
    i = np.argmin(abs(resonance_times-t))
    print(f"t={resonance_times[i]:g} Myr: nu5={resonances.nu5_roots_au[i]}, "
          f"nu6={resonances.nu6_roots_au[i]}")

## Direct N-body short run

This default 500-year experiment checks the complete force path. `WHFast` uses a fixed 0.025-year step. A separate 100-year comparison found this step close to `IAS15`; 0.1 year produced noticeably different eccentricities. The paper-specific compiled force agrees with the Python force callback over the same 100-year interval. The first run compiles a C shared library in the system temporary directory.

In [ ]:
short_end_myr = 5e-4   # 500 years; adjust after short-run QA
output_times_myr = np.linspace(0, short_end_myr, 101)
integration = IntegrationConfig(integrator="whfast", initial_step_years=0.025,
                                force_backend="compiled")
print(f"Estimated fixed steps: {integration.estimated_fixed_steps(short_end_myr):,}")
run = integrate_fig2(system, disk, gravity, drag,
                     output_times_myr=output_times_myr, integration=integration,
                     boundaries=boundaries)
fig, ax = plt.subplots(figsize=(9, 5.2))
plot_fig2_style(run, resonances=None, ax=ax)
ax.set_title("First 500 years: direct planetary gravity + disk gravity + drag")
fig.tight_layout()
plt.show()
for name in run.names:
    idx = run.names.index(name)
    print(f"{name}: a0={run.a_au[0,idx]:.6f} AU, a_end={run.a_au[-1,idx]:.6f} AU, "
          f"e_end={run.e[-1,idx]:.5f}")

## Saved 10 Myr trajectory for the selected configuration

This cell reads saved data only. It checks disk, gravity, drag, initial conditions and integration settings against the current selection. A mismatch is reported instead of showing a stale figure. The historical paper-disk file used a **100 km** body and is rejected under the notebook's 10 km settings. The displayed `gas_profile_baseline` 10 Myr track is exploratory: a 0.1 Myr step-size check differed by about 0.015 AU for the first particle, so its late-time values are not established as converged. For a new disk or initial condition, run the final integration cell first, then rerun this display cell.

In [ ]:
from paper_fig2_integrator import Fig2Run

saved_run_path = HERE / "outputs" / f"{experiment_label}_10myr.npz"
if saved_run_path.exists():
    saved_run = Fig2Run.load_npz(saved_run_path)
    if matches_configuration(saved_run, system, disk, gravity, drag, integration, boundaries):
        fig, ax = plt.subplots(figsize=(11, 6))
        plot_fig2_style(saved_run, resonances, ax=ax)
        ax.set_xlim(0.1, 10)
        ax.set_title(f"Exploratory 10 Myr evolution: {experiment_label}")
        fig.tight_layout()
        plt.show()
        for j, name in enumerate(saved_run.names):
            if name.startswith("particle_"):
                print(f"{name}: a0={saved_run.a_au[0,j]:.6f} AU, "
                      f"a10Myr={saved_run.a_au[-1,j]:.6f} AU, "
                      f"e10Myr={saved_run.e[-1,j]:.5f}")
        if saved_run.events:
            print("Boundary events:")
            for event in saved_run.events:
                print(f"  {event.particle_name}: {event.event_type}, "
                      f"t={event.time_myr:.6f} Myr, r={event.radius_au:.6f} AU, "
                      f"E={event.kepler_energy_code:.6g}")
    else:
        print(f"Saved data at {saved_run_path} do not match current settings; regenerate below.")
else:
    print(f"No saved 10 Myr data at {saved_run_path}; run the optional cell below.")

## Optional: integrate the selected disk for 10 Myr

Set `run_full_experiment=True` to generate new data. The compiled backend supports both disk providers. One 10 Myr run at 0.025 yr took about six minutes on this machine. The result is exploratory: the softened ring gravity is not the paper's Ward series, and long-run convergence and phase sensitivity remain open. Each disk choice saves to a different path. Repeating a run for the **same** disk choice overwrites that choice's file; copy or rename it first to retain multiple initial-condition trials.

In [ ]:
run_full_experiment = False
if run_full_experiment:
    full_times_myr = np.r_[0.0, np.geomspace(0.001, 10.0, 2500)]
    print(f"Estimated fixed steps: {integration.estimated_fixed_steps(10.0):,}")
    full_run = integrate_fig2(
        system, disk, gravity, drag,
        output_times_myr=full_times_myr,
        integration=integration,
        boundaries=boundaries,
    )
    result_path = HERE / "outputs" / f"{experiment_label}_10myr.npz"
    full_run.save_npz(result_path)
    fig, ax = plt.subplots(figsize=(11, 6))
    plot_fig2_style(full_run, resonances, ax=ax)
    ax.set_xlim(0.1, 10)
    ax.set_title(f"Exploratory 10 Myr evolution: {experiment_label}")
    fig.tight_layout()
    plt.show()
    print("Saved:", result_path)
    if full_run.events:
        print("Boundary events:")
        for event in full_run.events:
            print(f"  {event.particle_name}: {event.event_type}, "
                  f"t={event.time_myr:.6f} Myr, r={event.radius_au:.6f} AU, "
                  f"E={event.kepler_energy_code:.6g}")
    else:
        print("No particle boundary events.")